Outage Analysis

**Name(s)**: Cameron Chen

**Website Link**: (your website link)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import plotly.express as px
pd.options.plotting.backend = 'plotly'

# from dsc80_utils import * # Feel free to uncomment and use this.

ModuleNotFoundError: No module named 'plotly'

In [ ]:
!pip install openpyxl

## Step 1: Introduction

In [ ]:
# TODO
# What factors affect outage severity (time)?

# Load outage dataset (actual header row starts at row 6)
df = pd.read_excel("outage.xlsx", header=5)

# View dataset dimensions
df.shape

In [ ]:
df.head()

## Step 2: Data Cleaning and Exploratory Data Analysis

In [ ]:
# Clean column names for consistency
df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(" ", "_")
df.columns = df.columns.str.upper()

# Remove non-data row where OBS is missing
df = df[df["OBS"].notna()].reset_index(drop=True)

# Check updated shape
df.shape

In [ ]:
# Preview outage start date/time columns
df[["OUTAGE.START.DATE", "OUTAGE.START.TIME"]].head()

In [ ]:
# Columns that should be numeric
numeric_cols = [
    "YEAR", "MONTH",
    "ANOMALY.LEVEL",
    "CUSTOMERS.AFFECTED",
    "DEMAND.LOSS.MW",
    "POPPCT_URBAN"
]

# Convert to numeric (invalid values -> NaN)
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df["OUTAGE.START.DATE"]


In [ ]:
# Combine start date and time into one datetime column
df["OUTAGE.START"] = pd.to_datetime(
    df["OUTAGE.START.DATE"].astype(str) + " " +
    df["OUTAGE.START.TIME"].astype(str),
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

# Combine restoration date and time into datetime
df["OUTAGE.RESTORATION"] = pd.to_datetime(
    df["OUTAGE.RESTORATION.DATE"].astype(str) + " " +
    df["OUTAGE.RESTORATION.TIME"].astype(str),
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)

In [ ]:
# Create start datetime if both date and time exist
df["OUTAGE.START"] = df.apply(
    lambda row: datetime.combine(
        row["OUTAGE.START.DATE"].date(),
        row["OUTAGE.START.TIME"]
    ) if pd.notna(row["OUTAGE.START.DATE"]) and pd.notna(row["OUTAGE.START.TIME"]) else pd.NaT,
    axis=1
)

# Create restoration datetime if both date and time exist
df["OUTAGE.RESTORATION"] = df.apply(
    lambda row: datetime.combine(
        row["OUTAGE.RESTORATION.DATE"].date(),
        row["OUTAGE.RESTORATION.TIME"]
    ) if pd.notna(row["OUTAGE.RESTORATION.DATE"]) and pd.notna(row["OUTAGE.RESTORATION.TIME"]) else pd.NaT,
    axis=1
)

In [ ]:
# Calculate outage duration in hours
df["DURATION.HOURS"] = (
    df["OUTAGE.RESTORATION"] - df["OUTAGE.START"]
).dt.total_seconds() / 3600

In [ ]:
# Remove negative or missing durations
df = df[df["DURATION.HOURS"] > 0]
df = df.dropna(subset=["DURATION.HOURS"])

In [ ]:
# Median outage duration
median_duration = df["DURATION.HOURS"].median()

# Binary severity label (1 = above median)
df["HIGH.SEVERITY"] = (
    df["DURATION.HOURS"] > median_duration
).astype(int)

# Check class distribution
df["HIGH.SEVERITY"].value_counts(normalize=True)

In [ ]:
df["MONTH"] = df["OUTAGE.START"].dt.month

# Convert month to season category
df["SEASON"] = df["MONTH"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
})

In [ ]:
df.head()

In [ ]:
df.columns

The dataset was provided as an Excel file with metadata rows preceding the actual column headers. I reloaded the dataset using the correct header row and removed the units row. I standardized column names and converted relevant numeric columns to appropriate data types. I combined outage start and restoration date/time columns into timestamp variables and computed outage duration in hours. I removed invalid or negative durations and created a binary target variable, HIGH.SEVERITY, defined as outages lasting longer than the median duration. Finally, I engineered time-based features (day of week and season) and removed post-restoration variables to avoid data leakage.

Plot 1 - Distribution of Outage Duration

In [ ]:
fig = px.histogram(
    df,
    x="DURATION.HOURS",
    nbins=50,
    title="Distribution of Outage Duration (Hours)"
)

fig.update_layout(
    xaxis_title="Duration (Hours)",
    yaxis_title="Count"
)

fig.show()

In [ ]:
cause_counts = df["CAUSE.CATEGORY"].value_counts().reset_index()
cause_counts.columns = ["CAUSE.CATEGORY", "COUNT"]

fig = px.bar(
    cause_counts,
    x="CAUSE.CATEGORY",
    y="COUNT",
    title="Distribution of Outage Causes"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
fig = px.box(
    df,
    x="CAUSE.CATEGORY",
    y="DURATION.HOURS",
    title="Outage Duration by Cause"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
severity_by_season = (
    df.groupby("SEASON")["HIGH.SEVERITY"]
      .mean()
      .reset_index()
)

fig = px.bar(
    severity_by_season,
    x="SEASON",
    y="HIGH.SEVERITY",
    title="Proportion of High Severity Outages by Season"
)

fig.update_layout(yaxis_title="Proportion")
fig.show()

In [ ]:
severity_by_region = (
    df.groupby("CLIMATE.REGION")["HIGH.SEVERITY"]
      .mean()
      .reset_index()
      .sort_values(by="HIGH.SEVERITY", ascending=False)
)

fig = px.bar(
    severity_by_region,
    x="CLIMATE.REGION",
    y="HIGH.SEVERITY",
    title="High Severity Rate by Climate Region"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Step 3: Assessment of Missingness

In [ ]:
# Indicator for whether CUSTOMERS.AFFECTED is missing
df["CA_MISSING"] = df["CUSTOMERS.AFFECTED"].isna()

# Inspect the missingness indicator column
df["CA_MISSING"]

In [ ]:
# Observed difference in mean ANOMALY.LEVEL between missing vs non-missing groups
observed_diff = (
    df.groupby("CA_MISSING")["ANOMALY.LEVEL"]
    .mean()
    .diff()
    .iloc[-1]
)

In [ ]:
# Number of permutations
n = 1000
permuted_diffs = []

# Generate permutation distribution
for _ in range(n):
    shuffled = np.random.permutation(df["CA_MISSING"])
    perm_diff = (
        df.assign(shuffled=shuffled)
        .groupby("shuffled")["ANOMALY.LEVEL"]
        .mean()
        .diff()
        .iloc[-1]
    )
    permuted_diffs.append(perm_diff)

# Compute two-sided p-value
p_value = np.mean(np.abs(permuted_diffs) >= abs(observed_diff))

# Display p-value
p_value

In [ ]:
fig = px.histogram(
    x=permuted_diffs,
    nbins=40,
    title="Permutation Distribution of Mean Anomaly Difference"
)

fig.add_vline(x=observed_diff, line_color="red")
fig.show()

## Step 4: Hypothesis Testing

In [ ]:
# Outage durations caused by severe weather
weather = df[df["CAUSE.CATEGORY"] == "severe weather"]["DURATION.HOURS"]

# Outage durations caused by other factors
non_weather = df[df["CAUSE.CATEGORY"] != "severe weather"]["DURATION.HOURS"]

# Observed difference in mean outage duration
observed_stat = weather.mean() - non_weather.mean()

# Display observed statistic
observed_stat

In [ ]:
# Combine both groups into one array
combined = np.concatenate([weather, non_weather])

# Size of the weather group
n_weather = len(weather)

# Number of permutations
n = 1000
permuted_stats = []

# Generate permutation distribution
for _ in range(n):
    np.random.shuffle(combined)
    perm_weather = combined[:n_weather]
    perm_non_weather = combined[n_weather:]
    perm_stat = perm_weather.mean() - perm_non_weather.mean()
    permuted_stats.append(perm_stat)

# Compute p-value
p_value = np.mean(np.array(permuted_stats) >= observed_stat)

# Display p-value
p_value

In [ ]:


fig = px.histogram(
    x=permuted_stats,
    nbins=40,
    title="Permutation Distribution of Mean Duration Difference"
)

fig.add_vline(x=observed_stat, line_color="red")

fig.update_layout(
    xaxis_title="Mean Difference (Weather - Non-Weather)",
    yaxis_title="Frequency"
)

fig.show()

## Step 5: Framing a Prediction Problem

In [ ]:
# Response variable for prediction
response = "HIGH.SEVERITY"

# Display response information
print("Response variable:", response)
print("Unique values:", df[response].dropna().unique())
print("Problem type: Classification")

# Candidate predictor variables
features = [
    "CAUSE.CATEGORY",
    "CLIMATE.REGION",
    "SEASON",
    "ANOMALY.LEVEL",
    "POPPCT_URBAN",
    "YEAR"
]

# Display selected feature columns
print("\nCandidate predictor columns:")
print(features)

# Check for any missing columns
print("\nMissing selected columns from df:")
print([col for col in features + [response] if col not in df.columns])

# Create modeling dataframe with selected columns
model_df = df[features + [response]].dropna().copy()

# Preview modeling dataset
model_df.head()

## Step 6: Baseline Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Baseline feature set
features = ["ANOMALY.LEVEL", "POPPCT_URBAN", "CAUSE.CATEGORY"]
target = "HIGH.SEVERITY"

# Remove rows with missing values
model_df = df[features + [target]].dropna()

# Separate predictors and target
X = model_df[features]
y = model_df[target]

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Define numeric and categorical features
numeric_features = ["ANOMALY.LEVEL", "POPPCT_URBAN"]
categorical_features = ["CAUSE.CATEGORY"]

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Baseline model pipeline
baseline_model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train baseline model
baseline_model.fit(X_train, y_train)

# Generate predictions
y_pred = baseline_model.predict(X_test)

# Compute accuracy
accuracy = accuracy_score(y_test, y_pred)
accuracy

## Step 7: Final Model

In [ ]:
# Import tools for hyperparameter tuning and feature transformation
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import FunctionTransformer

# Feature engineering: absolute anomaly level
df["ANOMALY_ABS"] = df["ANOMALY.LEVEL"].abs()

# Feature engineering: binary indicator for highly urban areas
urban_median = df["POPPCT_URBAN"].median()
df["URBAN_HIGH"] = (df["POPPCT_URBAN"] > urban_median).astype(int)

# Final feature set
features_final = [
    "ANOMALY.LEVEL",
    "ANOMALY_ABS",
    "POPPCT_URBAN",
    "URBAN_HIGH",
    "CAUSE.CATEGORY"
]

# Target variable
target = "HIGH.SEVERITY"

# Create modeling dataset
model_df = df[features_final + [target]].dropna()

# Separate predictors and response
X = model_df[features_final]
y = model_df[target]

# Use same train/test split as baseline model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Numeric features to scale
numeric_features = [
    "ANOMALY.LEVEL",
    "ANOMALY_ABS",
    "POPPCT_URBAN"
]

# Categorical features to encode
categorical_features = [
    "CAUSE.CATEGORY",
    "URBAN_HIGH"
]

# Preprocessing: scaling + one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Modeling pipeline
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Hyperparameter search space
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

# Grid search with cross-validation
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy"
)

# Train models across parameter grid
grid.fit(X_train, y_train)

# Retrieve best model
best_model = grid.best_estimator_

# Generate predictions on test set
y_pred = best_model.predict(X_test)

# Evaluate final model accuracy
final_accuracy = accuracy_score(y_test, y_pred)

# Display best parameter and performance
print("Best C:", grid.best_params_)
print("Final model accuracy:", final_accuracy)

## Step 8: Fairness Analysis

In [ ]:
# Import evaluation metric
from sklearn.metrics import accuracy_score

# Generate predictions from final model
y_pred = best_model.predict(X_test)

# Store predictions and true labels
results = X_test.copy()
results["true"] = y_test.values
results["pred"] = y_pred

# Define group indicator (East North Central vs others)
results["GROUP"] = df.loc[X_test.index, "CLIMATE.REGION"] == "East North Central"

# Compute model accuracy for each group
group_acc = results.groupby("GROUP").apply(
    lambda x: accuracy_score(x["true"], x["pred"])
)

# Observed difference in group accuracy
observed_diff = group_acc.diff().iloc[-1]

observed_diff

In [ ]:
# Number of permutations
n = 1000
permuted_diffs = []

# Generate permutation distribution
for _ in range(n):

    # Shuffle group labels
    shuffled = np.random.permutation(results["GROUP"])

    perm = results.assign(shuffled_group=shuffled)

    # Compute accuracy for shuffled groups
    acc = perm.groupby("shuffled_group").apply(
        lambda x: accuracy_score(x["true"], x["pred"])
    )

    # Difference in accuracy between groups
    diff = acc.diff().iloc[-1]

    permuted_diffs.append(diff)

# Compute two-sided p-value
p_value = np.mean(np.abs(permuted_diffs) >= abs(observed_diff))

p_value

In [ ]:
fig = px.histogram(
    x=permuted_diffs,
    nbins=40,
    title="Permutation Distribution of Accuracy Difference"
)

fig.add_vline(x=observed_diff, line_color="red")

fig.show()